# RAG Benchmark – Results Analysis

This notebook builds a **per-question metrics DataFrame** from the benchmark outputs,
aggregates by configuration and by individual pipeline component (Chunker, Embedder, LLM),
and produces plots for the thesis.

**Expected input files in `evaluation/results/`**

| File | Created by |
|---|---|
| `evaluation_dataset.json` | `orchestration/benchmark_loop.py` |
| `per_question_scores.json` | `evaluation/ragas_eval.py` |

Run from the **project root** (default CWD in VS Code).

In [ ]:
import json
import warnings
from itertools import combinations
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# ── Pipeline dimension names & metric lists ────────────────────────────────
COMPONENTS    = ['Chunker', 'Embedder', 'LLM']
RAGAS_METRICS = ['faithfulness', 'answer_relevancy', 'context_recall', 'context_precision']
ALL_METRICS   = ['latency', 'source_attribution'] + RAGAS_METRICS

METRIC_LABELS = {
    'faithfulness':       'Faithfulness',
    'answer_relevancy':   'Answer Relevancy',
    'context_recall':     'Context Recall',
    'context_precision':  'Context Precision',
    'source_attribution': 'Source Attribution',
    'latency':            'Latency (s)',
    'prompt_tokens':      'Prompt Tokens',
    'completion_tokens':  'Completion Tokens',
}

# ── Thresholds (used in Sections 9 & 12) ─────────────────────────────────
# Grounded in RAGAS documentation and RAG evaluation literature:
#   Faithfulness >= 0.80  → well-grounded; < 0.60 → hallucination risk
#   Answer Relevancy >= 0.80 → on-topic; < 0.60 → off-topic
#   Context Recall >= 0.70  → most evidence retrieved; < 0.50 → missing evidence
#   Context Precision >= 0.50 → majority of chunks relevant; < 0.30 → noisy retrieval
#   Source Attribution >= 0.80 → cited in 80%+ of answers (ESG traceability)
#   Latency <= 10s → interactive; <= 20s → acceptable for research/batch use
THRESHOLDS = {
    'faithfulness':       {'good': 0.80, 'acceptable': 0.60},
    'answer_relevancy':   {'good': 0.80, 'acceptable': 0.60},
    'context_recall':     {'good': 0.70, 'acceptable': 0.50},
    'context_precision':  {'good': 0.50, 'acceptable': 0.30},
    'source_attribution': {'good': 0.80, 'acceptable': 0.60},
    'latency':            {'good': 10.0, 'acceptable': 20.0, 'direction': 'lower_is_better'},
}


def classify(val, metric):
    """Classify a single metric value as good / acceptable / poor."""
    if pd.isna(val):
        return 'missing'
    thr = THRESHOLDS.get(metric, {})
    direction = thr.get('direction', 'higher_is_better')
    good, acc = thr.get('good', float('nan')), thr.get('acceptable', float('nan'))
    if direction == 'higher_is_better':
        return 'good' if val >= good else ('acceptable' if val >= acc else 'poor')
    return 'good' if val <= good else ('acceptable' if val <= acc else 'poor')


## 1  Load raw data

In [ ]:
assert Path('../evaluation/results/evaluation_dataset.json').exists(), (
    'Not found: ./evaluation/results/evaluation_dataset.json\nRun orchestration/benchmark_loop.py first.'
)

with open('../evaluation/results/evaluation_dataset.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

df_raw = pd.DataFrame(raw_data)

print(
    f"Loaded {len(df_raw):,} records  "
    f"| {df_raw['Configuration'].nunique()} configuration(s)  "
    f"| {df_raw['question_id'].nunique()} question(s)"
)
df_raw[['question_id', 'Configuration', 'latency', 'prompt_tokens', 'completion_tokens']].head()

In [ ]:
df_raw['source_attribution'] = df_raw.apply(
    lambda r: 1.0 if str(r.get('expected_source', '')) in str(r.get('answer', '')) else 0.0,
    axis=1,
)

In [ ]:
for m in RAGAS_METRICS:
    df_raw[m] = float('nan')

if Path('../evaluation/results/per_question_scores.json').exists():
    df_ragas = pd.read_json('../evaluation/results/per_question_scores.json')
    available = [c for c in RAGAS_METRICS if c in df_ragas.columns]
    if available:
        df_raw = df_raw.drop(columns=available, errors='ignore')
        df_raw = df_raw.merge(
            df_ragas[['question_id', 'Configuration'] + available].drop_duplicates(),
            on=['question_id', 'Configuration'],
            how='left',
        )
        print(f'Merged RAGAS columns: {available}')
    else:
        print('per_question_scores.json found but contains no RAGAS metric columns.')
else:
    print('per_question_scores.json not found – RAGAS metrics are NaN.\nRun: python evaluation/ragas_eval.py')

df = df_raw.drop(columns=['contexts', 'answer', 'ground_truth'], errors='ignore').copy()
print(f'\nMaster DataFrame: {len(df)} rows x {len(df.columns)} columns')
df.dtypes

## 2  Per-question view

One row per (question x configuration).
This is the ground-truth dataset for all subsequent analysis.

In [ ]:
display_cols = (
    [c for c in ['question_id', 'question', 'Configuration', 'Chunker', 'Embedder', 'LLM'] if c in df.columns]
    + [m for m in ALL_METRICS if m in df.columns]
)
df[display_cols].sort_values(['question_id', 'Configuration']).reset_index(drop=True)

In [ ]:
n_questions = df['question_id'].nunique()
n_configs   = df['Configuration'].nunique()
counts      = df.groupby('Configuration')['question_id'].count()

if counts.nunique() > 1:
    print('WARNING – unequal question counts per configuration:')
    print(counts[counts != n_questions].to_string())
else:
    print(f'OK – {n_configs} configurations x {n_questions} question(s) = {len(df)} rows')

metric_cols = [m for m in ALL_METRICS if m in df.columns]
missing     = df[metric_cols].isna().sum()
if missing.any():
    print('\nMissing values per metric:')
    print(missing[missing > 0].to_string())
else:
    print('\nNo missing metric values.')

## 3  Configuration-level leaderboard

Each row is the **mean across all questions** for that configuration.

In [ ]:
metric_cols = [m for m in ALL_METRICS if m in df.columns]

df_configs = (
    df.groupby(['Configuration', 'Chunker', 'Embedder', 'LLM'])
    .agg(
        n_questions=('question_id', 'count'),
        **{m: (m, 'mean') for m in metric_cols},
    )
    .reset_index()
    .round(4)
)

ragas_in_configs = [m for m in RAGAS_METRICS if m in df_configs.columns and df_configs[m].notna().any()]
if ragas_in_configs:
    df_configs['quality_score'] = df_configs[ragas_in_configs].mean(axis=1)
    df_configs = df_configs.sort_values('quality_score', ascending=False)
else:
    df_configs = df_configs.sort_values('source_attribution', ascending=False)

df_configs = df_configs.reset_index(drop=True)
df_configs.to_csv('config_leaderboard.csv', index=False, sep=';')
print(f'Saved config_leaderboard.csv  ({len(df_configs)} configurations)')
df_configs

## 4  Component-level analysis

For each pipeline dimension (Chunker, Embedder, LLM) we average over **all configurations
that share that component value** (9 configs per value in a full 3x3x3 design).
This gives the marginal effect of each choice, independent of the others.

In [ ]:
dfs_comp = {}
for comp in COMPONENTS:
    grp = (
        df.groupby(comp)
        .agg(
            n_samples=('question_id', 'count'),
            **{m: (m, 'mean') for m in metric_cols},
        )
        .reset_index()
        .round(4)
    )
    dfs_comp[comp] = grp
    grp.to_csv(f'component_{comp.lower()}.csv', index=False, sep=';')

for comp in COMPONENTS:
    print(f"\n{'=' * 60}")
    print(f'  By {comp}')
    print(f"{'=' * 60}")
    display(dfs_comp[comp])

## 5  Visualisations

### 5a–5f  Overview graphs

In [ ]:
sns.set_theme(style='whitegrid', font_scale=1.05)
PALETTE = sns.color_palette('tab10')
METRIC_COLORS = {m: PALETTE[i] for i, m in enumerate(ALL_METRICS)}

In [ ]:
# 5a  Heatmap: configurations x quality + attribution metrics
hm_metrics = ['source_attribution'] + [m for m in RAGAS_METRICS if m in df_configs.columns]
plot_data = df_configs.set_index('Configuration')[hm_metrics].copy()
plot_data.columns = [METRIC_LABELS.get(c, c) for c in hm_metrics]
normed = plot_data.apply(lambda col: (col - col.min()) / ((col.max() - col.min()) or 1))
annot  = plot_data.round(3).astype(str).replace('nan', '-')

fig, ax = plt.subplots(figsize=(max(8, 2.5 * len(hm_metrics)), max(6, 0.45 * len(df_configs))))
sns.heatmap(normed, annot=annot, fmt='', cmap='RdYlGn',
            linewidths=0.4, linecolor='#ddd',
            cbar_kws={'label': 'Relative score (normalised per metric)'}, ax=ax)
ax.set_title('RAG configuration leaderboard – quality metrics', pad=12)
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig('heatmap_configs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5b  Bar charts: each component x quality metrics
quality_metrics = ['source_attribution'] + [m for m in RAGAS_METRICS if m in df.columns]
n_rows = len(COMPONENTS)
n_cols = len(quality_metrics)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(max(12, 3.5 * n_cols), 3.5 * n_rows), sharey=False)
if n_rows == 1:
    axes = axes[None, :]

for row_i, comp in enumerate(COMPONENTS):
    grp = dfs_comp[comp]
    for col_j, metric in enumerate(quality_metrics):
        ax    = axes[row_i, col_j]
        vals  = grp[metric]
        color = METRIC_COLORS.get(metric, '#4C72B0')
        bars  = ax.bar(grp[comp], vals, color=color, alpha=0.80, edgecolor='white', zorder=3)
        valid = vals.dropna()
        y_max = max(valid.max() * 1.30 if not valid.empty else 1.0, 0.05)
        ax.set_ylim(0, y_max)
        ax.set_title(METRIC_LABELS.get(metric, metric) if row_i == 0 else '', fontsize=9.5)
        if col_j == 0:
            ax.set_ylabel(comp, fontsize=9.5)
        ax.tick_params(axis='x', labelrotation=15, labelsize=8)
        ax.yaxis.grid(True, linestyle='--', alpha=0.5)
        ax.set_axisbelow(True)
        for bar, val in zip(bars, vals):
            if not pd.isna(val):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + y_max * 0.02,
                        f'{val:.2f}', ha='center', va='bottom', fontsize=7.5)

fig.suptitle('Mean quality metrics by pipeline component', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('barplot_components_quality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5c  Bar charts: each component x latency
fig, axes = plt.subplots(1, len(COMPONENTS), figsize=(4.5 * len(COMPONENTS), 4.5), sharey=False)
for col_j, comp in enumerate(COMPONENTS):
    ax   = axes[col_j]
    grp  = dfs_comp[comp]
    vals = grp['latency']
    color = METRIC_COLORS.get('latency', '#937860')
    bars  = ax.bar(grp[comp], vals, color=color, alpha=0.80, edgecolor='white', zorder=3)
    valid = vals.dropna()
    y_max = valid.max() * 1.30 if not valid.empty else 1.0
    ax.set_ylim(0, y_max)
    ax.set_title(comp, fontsize=10)
    ax.set_ylabel('Latency (s)' if col_j == 0 else '')
    ax.tick_params(axis='x', labelrotation=15, labelsize=8)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)
    for bar, val in zip(bars, vals):
        if not pd.isna(val):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + y_max * 0.02,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=7.5)

fig.suptitle('Mean latency by pipeline component', fontsize=13)
plt.tight_layout()
plt.savefig('barplot_components_latency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5d  Scatter: latency vs average quality
ragas_in_configs = [m for m in RAGAS_METRICS if m in df_configs.columns and df_configs[m].notna().any()]
if ragas_in_configs:
    df_configs['avg_quality'] = df_configs[ragas_in_configs].mean(axis=1)
    y_label = 'Avg. RAGAS quality score'
else:
    df_configs['avg_quality'] = df_configs['source_attribution']
    y_label = 'Source Attribution'

llm_order  = sorted(df_configs['LLM'].unique())
llm_colors = {llm: PALETTE[i] for i, llm in enumerate(llm_order)}

fig, ax = plt.subplots(figsize=(7, 5))
for llm in llm_order:
    mask = df_configs['LLM'] == llm
    ax.scatter(df_configs.loc[mask, 'latency'], df_configs.loc[mask, 'avg_quality'],
               label=llm, color=llm_colors[llm], s=90, alpha=0.85, edgecolors='k', linewidths=0.4, zorder=3)
ax.set_xlabel(METRIC_LABELS.get('latency', 'latency'))
ax.set_ylabel(y_label)
ax.set_title('Latency vs Quality')
ax.legend(title='LLM', fontsize=8)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('scatter_latency_quality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5e  Box plot: latency distribution by LLM
order = df.groupby('LLM')['latency'].median().sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=df, x='LLM', y='latency', order=order, palette='Set2', ax=ax)
ax.set_title('Latency distribution by LLM')
ax.set_xlabel('')
ax.set_ylabel('Latency (s)')
plt.tight_layout()
plt.savefig('boxplot_latency_llm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5f  Correlation matrix (per-question level)
corr_cols = [m for m in ALL_METRICS if m in df.columns and df[m].notna().sum() >= 3]
if len(corr_cols) >= 2:
    corr   = df[corr_cols].corr()
    labels = [METRIC_LABELS.get(c, c) for c in corr_cols]
    mask   = np.triu(np.ones_like(corr, dtype=bool), k=1)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                mask=mask, xticklabels=labels, yticklabels=labels, linewidths=0.5, ax=ax)
    ax.set_title('Metric correlation matrix (per-question level)')
    plt.tight_layout()
    plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Not enough non-NaN metric columns for a correlation matrix yet.')

## 6  Token & Cost Analysis

Prompt tokens determine the **input cost** and the amount of context the LLM must process;
completion tokens determine the **output cost** and correlate with answer verbosity.
Both scale API costs linearly, so understanding which component drives token count helps
estimate the operational budget of each pipeline configuration.

**Why a grouped bar chart?** Placing prompt and completion bars side-by-side for each
component value lets you immediately see whether a difference is driven by longer contexts
(chunker/embedder effect) or by more verbose outputs (LLM effect).

**Expected pattern:** the LLM dimension should dominate completion tokens; chunker strategy
may shift prompt tokens because different chunkers produce different chunk sizes.

In [ ]:
# 6  Token usage by component
token_cols = [c for c in ['prompt_tokens', 'completion_tokens'] if c in df.columns]
if not token_cols:
    print('No token columns found – skipping.')
else:
    fig, axes = plt.subplots(1, len(COMPONENTS), figsize=(5.5 * len(COMPONENTS), 4.5))
    for col_j, comp in enumerate(COMPONENTS):
        ax  = axes[col_j]
        grp = df.groupby(comp)[token_cols].mean().reset_index()
        x   = np.arange(len(grp))
        w   = 0.35
        if 'prompt_tokens' in token_cols:
            ax.bar(x - w / 2, grp['prompt_tokens'], w, label='Prompt tokens',
                   color=PALETTE[0], alpha=0.82, edgecolor='white')
        if 'completion_tokens' in token_cols:
            ax.bar(x + w / 2, grp['completion_tokens'], w, label='Completion tokens',
                   color=PALETTE[1], alpha=0.82, edgecolor='white')
        ax.set_xticks(x)
        ax.set_xticklabels([str(v).split('/')[-1][:20] for v in grp[comp]],
                           rotation=15, ha='right', fontsize=8)
        ax.set_title(comp, fontsize=10)
        ax.set_ylabel('Mean token count' if col_j == 0 else '')
        ax.yaxis.grid(True, linestyle='--', alpha=0.5)
        ax.set_axisbelow(True)
        if col_j == 0:
            ax.legend(fontsize=8)
    fig.suptitle('Mean token usage by pipeline component', fontsize=13)
    plt.tight_layout()
    plt.savefig('barplot_token_usage.png', dpi=150, bbox_inches='tight')
    plt.show()

    token_summary = df.groupby('LLM')[token_cols].agg(['mean', 'std']).round(1)
    print('\nToken usage by LLM (mean ± std):')
    display(token_summary)

## 7  Distribution Analysis – Violin Plots

Bar charts in Section 5b show only the **mean**, hiding variability.
A configuration with a high mean but high variance may fail on many individual
questions even if the average looks good – a serious risk in production.

**Why violin plots?**
They overlay a **kernel density estimate** on top of a box plot, showing the full
shape of the distribution. This reveals:
- **Consistency:** a narrow violin = reliable across questions.
- **Bimodality:** two distinct bulges = the component performs well on one question
  type but poorly on another (e.g., short vs. long answers).
- **Floor/ceiling effects:** mass piled at 0 or 1 (common for faithfulness and
  source attribution) would be invisible in a bar chart.

Dashed threshold lines (green = good, orange = acceptable) let you read off the
fraction of the distribution that falls above each tier directly from the chart.

In [ ]:
# 7  Violin plots: metric x component
violin_metrics = [m for m in RAGAS_METRICS + ['source_attribution'] if m in df.columns]
n_rows = len(COMPONENTS)
n_cols = len(violin_metrics)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(max(14, 3.2 * n_cols), 3.5 * n_rows), sharey=False)
if n_rows == 1:
    axes = axes[None, :]

for row_i, comp in enumerate(COMPONENTS):
    for col_j, metric in enumerate(violin_metrics):
        ax    = axes[row_i, col_j]
        order = df.groupby(comp)[metric].median().sort_values(ascending=False).index.tolist()
        sns.violinplot(data=df, x=comp, y=metric, order=order,
                       palette='Set2', ax=ax, inner='box', cut=0)
        if metric in THRESHOLDS and THRESHOLDS[metric].get('direction', 'h') != 'lower_is_better':
            ax.axhline(THRESHOLDS[metric]['good'],       color='green',      linestyle='--', alpha=0.6, lw=1)
            ax.axhline(THRESHOLDS[metric]['acceptable'], color='darkorange', linestyle='--', alpha=0.6, lw=1)
        ax.set_title(METRIC_LABELS.get(metric, metric) if row_i == 0 else '', fontsize=9)
        ax.set_xlabel('')
        ax.set_ylabel(comp if col_j == 0 else '')
        ax.tick_params(axis='x', labelrotation=15, labelsize=7)
        ax.set_ylim(-0.05, 1.10)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4)

fig.suptitle(
    'Score distributions by pipeline component (violin plots)\n'
    'Dashed lines: green = good threshold, orange = acceptable threshold',
    fontsize=12, y=1.02,
)
plt.tight_layout()
plt.savefig('violin_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 8  Radar Charts (Multi-metric Profiles)

A radar / spider chart maps each metric onto its own axis radiating from the centre
and fills the enclosed polygon for each entity being compared.

**Why radar charts here?**
- They allow comparison of **all five quality dimensions simultaneously** in one glance.
- A lopsided polygon immediately flags a configuration that excels on one metric but
  underperforms on another (e.g., high faithfulness but low context recall).
- Overlaying multiple configurations exposes trade-offs invisible in per-metric bar charts.

**Important limitation:** the visual area of a polygon is not linearly proportional to
the underlying scores, and the ordering of axes affects perceived similarity.
Use these charts for **pattern recognition** (shape + imbalance), not precise comparison –
always cross-check with the numerical tables.

**Two views produced:**
1. Top-6 configurations (by RAGAS quality score)
2. Averaged profile per component value (one chart per dimension: Chunker, Embedder, LLM)

In [ ]:
# 8  Radar charts
radar_metrics = [m for m in RAGAS_METRICS + ['source_attribution'] if m in df_configs.columns]
radar_labels  = [METRIC_LABELS.get(m, m) for m in radar_metrics]
N      = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]


def radar_fill(ax, values, color, label, alpha=0.22):
    vals = list(values) + [values[0]]
    ax.plot(angles, vals, color=color, linewidth=1.8, label=label)
    ax.fill(angles, vals, color=color, alpha=alpha)


# 8a  Top-6 configurations
top_n = min(6, len(df_configs))
fig, axes = plt.subplots(2, 3, figsize=(15, 9), subplot_kw=dict(projection='polar'))
axes = axes.flatten()
for i, (_, row) in enumerate(df_configs.head(top_n).iterrows()):
    ax   = axes[i]
    vals = [max(0.0, min(1.0, row.get(m, 0) or 0)) for m in radar_metrics]
    radar_fill(ax, vals, PALETTE[i], row['Configuration'])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, size=8)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=6)
    ax.set_title(row['Configuration'].replace(' | ', '\n'), size=7, pad=14)
    ax.grid(alpha=0.3)
for j in range(top_n, len(axes)):
    axes[j].set_visible(False)
fig.suptitle(f'Radar charts – top {top_n} configurations (by RAGAS quality score)', fontsize=13)
plt.tight_layout()
plt.savefig('radar_top_configs.png', dpi=150, bbox_inches='tight')
plt.show()

# 8b  Averaged profile per component value
for comp in COMPONENTS:
    grp = dfs_comp[comp]
    fig, ax = plt.subplots(figsize=(7, 6), subplot_kw=dict(projection='polar'))
    for i, (_, row) in enumerate(grp.iterrows()):
        vals = [max(0.0, min(1.0, row.get(m, 0) or 0)) for m in radar_metrics]
        radar_fill(ax, vals, PALETTE[i], str(row[comp]).split('/')[-1])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, size=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=7)
    ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=8)
    ax.set_title(f'Metric profile by {comp}', fontsize=12, pad=16)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'radar_by_{comp.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9  Threshold Analysis & Pass Rates

Rather than asking *which config has the highest mean?*, threshold analysis asks:
*which config reliably meets a minimum acceptable standard for each individual question?*

A configuration with a high mean but frequent individual failures is riskier in production
than one with a slightly lower mean but consistent above-threshold performance.

### Threshold justification

| Metric | Good | Acceptable | Rationale |
|---|---|---|---|
| Faithfulness | >= 0.80 | >= 0.60 | RAGAS docs; below 0.60 = hallucination risk |
| Answer Relevancy | >= 0.80 | >= 0.60 | RAGAS docs; below 0.60 = off-topic answer |
| Context Recall | >= 0.70 | >= 0.50 | RAG literature; below 0.50 = missing evidence |
| Context Precision | >= 0.50 | >= 0.30 | Below 0.30 = mostly irrelevant chunks |
| Source Attribution | >= 0.80 | >= 0.60 | ESG traceability; source must be cited |
| Latency | <= 10s | <= 20s | Interactive: 10s; research/batch: 20s |

**Charts produced:**
1. **Pass-rate heatmap** – per configuration, fraction of questions reaching the *good* tier
2. **Stacked bars per component** – good / acceptable / poor breakdown for each metric

In [ ]:
# 9a  Pass-rate heatmap by configuration
threshold_metrics = [m for m in THRESHOLDS if m in df.columns]
df_class = df.copy()
for m in threshold_metrics:
    df_class[f'{m}_class'] = df_class[m].apply(lambda v: classify(v, m))

pass_rates = {}
for m in threshold_metrics:
    pass_rates[m] = (
        df_class.groupby('Configuration')[f'{m}_class']
        .apply(lambda s: (s == 'good').mean())
    )

df_pass = pd.DataFrame(pass_rates)
df_pass.columns = [METRIC_LABELS.get(c, c) for c in df_pass.columns]
df_pass = df_pass.reindex(df_configs['Configuration'].values).dropna(how='all')

fig, ax = plt.subplots(figsize=(max(8, 2.5 * len(threshold_metrics)), max(6, 0.45 * len(df_pass))))
annot_pass = df_pass.map(lambda v: f'{v:.0%}' if not pd.isna(v) else '-')
sns.heatmap(df_pass, annot=annot_pass, fmt='', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.4, linecolor='#ddd',
            cbar_kws={'label': "Fraction of questions meeting 'good' threshold"}, ax=ax)
ax.set_title(
    "Pass rate (>= 'good' threshold) per configuration\n"
    'Configs ordered by overall RAGAS quality score (best at top)',
    pad=12,
)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig('heatmap_pass_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 9b  Stacked bars: pass / acceptable / poor per component
colors_stack = {'good': '#2ca02c', 'acceptable': '#ff7f0e', 'poor': '#d62728'}
for m in threshold_metrics:
    col = f'{m}_class'
    thr = THRESHOLDS[m]
    d   = thr.get('direction', 'higher_is_better')
    thr_label = (f"good >= {thr['good']}, acceptable >= {thr['acceptable']}"
                 if d == 'higher_is_better'
                 else f"good <= {thr['good']}s, acceptable <= {thr['acceptable']}s")

    fig, axes = plt.subplots(1, len(COMPONENTS), figsize=(5.5 * len(COMPONENTS), 4.5))
    for col_j, comp in enumerate(COMPONENTS):
        ax  = axes[col_j]
        grp = (
            df_class.groupby([comp, col]).size()
            .unstack(fill_value=0)
            .reindex(columns=['good', 'acceptable', 'poor'], fill_value=0)
        )
        grp_pct = grp.div(grp.sum(axis=1), axis=0)
        bottom = np.zeros(len(grp_pct))
        for cat in ['good', 'acceptable', 'poor']:
            if cat in grp_pct.columns:
                vals = grp_pct[cat].values
                ax.bar(range(len(grp_pct)), vals, bottom=bottom,
                       color=colors_stack[cat], label=cat, edgecolor='white')
                for xi, (v, b) in enumerate(zip(vals, bottom)):
                    if v > 0.06:
                        ax.text(xi, b + v / 2, f'{v:.0%}', ha='center', va='center',
                                fontsize=8, color='white', fontweight='bold')
                bottom += vals
        ax.set_xticks(range(len(grp_pct)))
        ax.set_xticklabels([str(v).split('/')[-1][:18] for v in grp_pct.index],
                           rotation=15, ha='right', fontsize=8)
        ax.set_ylim(0, 1.05)
        ax.set_title(comp, fontsize=10)
        ax.set_ylabel('Fraction of questions' if col_j == 0 else '')
        if col_j == 0:
            ax.legend(fontsize=8, loc='lower left')
    fig.suptitle(
        f'{METRIC_LABELS.get(m, m)} – pass / acceptable / poor breakdown by component\n({thr_label})',
        fontsize=12,
    )
    plt.tight_layout()
    plt.savefig(f'stacked_threshold_{m}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10  Two-Way Component Interaction Heatmaps

The component analysis in Section 4 reports **marginal** effects – it averages each
component value across all partners, implicitly assuming *independence* between dimensions.
In reality, one LLM might excel specifically when paired with a particular chunker
(an **interaction effect**), but that synergy would be invisible in a marginal mean.

**Why these heatmaps?**
A pivot-table heatmap for each pair (Chunker x Embedder, Chunker x LLM, Embedder x LLM)
makes interaction effects immediately visible:
- If all cells in a row are dark green, the *row component* dominates regardless of partner.
- If one isolated cell stands out, a specific *combination* drives the result – a true interaction.
- Uniform column colours suggest the *column component* dominates.

Both **composite quality score** and **latency** are shown per pair.

In [ ]:
# 10  Two-way interaction heatmaps
ragas_present = [m for m in RAGAS_METRICS if m in df.columns]
df_interact = df.copy()
df_interact['quality_score'] = df_interact[ragas_present].mean(axis=1)

pairs = [('Chunker', 'Embedder'), ('Chunker', 'LLM'), ('Embedder', 'LLM')]
plot_metrics_interact = [('quality_score', 'RdYlGn'), ('latency', 'RdYlGn_r')]

fig, axes = plt.subplots(
    len(pairs), len(plot_metrics_interact),
    figsize=(11 * len(plot_metrics_interact), 4.5 * len(pairs)),
)
for row_i, (comp_x, comp_y) in enumerate(pairs):
    for col_j, (metric, cmap) in enumerate(plot_metrics_interact):
        ax    = axes[row_i, col_j]
        pivot = df_interact.groupby([comp_x, comp_y])[metric].mean().unstack()
        pivot.index   = [str(v).split('/')[-1][:22] for v in pivot.index]
        pivot.columns = [str(v).split('/')[-1][:22] for v in pivot.columns]
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap=cmap, linewidths=0.5, ax=ax,
                    cbar_kws={'label': METRIC_LABELS.get(metric, metric)})
        ax.set_title(f'{comp_x} x {comp_y}  -  {METRIC_LABELS.get(metric, metric)}', fontsize=10)
        ax.set_xlabel(comp_y)
        ax.set_ylabel(comp_x)
        ax.tick_params(axis='x', labelrotation=20, labelsize=8)
        ax.tick_params(axis='y', labelrotation=0,  labelsize=8)

fig.suptitle('Two-way component interaction heatmaps', fontsize=14)
plt.tight_layout()
plt.savefig('heatmap_interactions.png', dpi=150, bbox_inches='tight')
plt.show()

## 11  Pareto Frontier – Quality vs Latency

No single configuration is 'best' on all dimensions simultaneously – faster models often
sacrifice quality, and high-quality configurations are usually slower.
The **Pareto frontier** identifies configurations that are *non-dominated*:
no other configuration achieves both higher quality *and* lower latency simultaneously.

**Why Pareto analysis?**
- It formally separates the efficient trade-off frontier from clearly suboptimal configurations.
- A configuration *off* the frontier can always be replaced by a Pareto-optimal one that
  is better on at least one dimension without being worse on the other.
- It makes the quality-speed trade-off explicit and quantifiable, supporting rational
  decision-making rather than arbitrary ranking.

**Reading the chart:**
- Large, opaque markers = Pareto-optimal configurations.
- Small, faded markers = dominated (avoidable) configurations.
- The dashed step line connects the Pareto-optimal points.
- Green horizontal line = minimum quality floor (0.75); orange vertical line = latency ceiling (15s).
- Configurations inside the top-left quadrant (green & orange lines) are **both fast and high-quality**.

In [ ]:
# 11  Pareto frontier
ragas_in_p = [m for m in RAGAS_METRICS if m in df_configs.columns and df_configs[m].notna().any()]
df_pareto = df_configs.copy()
df_pareto['avg_quality'] = df_pareto[ragas_in_p].mean(axis=1)


def pareto_efficient(quality, latency):
    """Return boolean mask: True if point i is not dominated (maximise quality, minimise latency)."""
    n, efficient = len(quality), np.ones(len(quality), dtype=bool)
    for i in range(n):
        if efficient[i]:
            dominated = (
                (quality >= quality[i]) & (latency <= latency[i]) &
                ((quality > quality[i]) | (latency < latency[i]))
            )
            dominated[i] = False
            efficient[efficient] = ~dominated[efficient]
    return efficient


df_pareto['pareto'] = pareto_efficient(
    df_pareto['avg_quality'].values, df_pareto['latency'].values
)

llm_order_p  = sorted(df_pareto['LLM'].unique())
llm_colors_p = {llm: PALETTE[i] for i, llm in enumerate(llm_order_p)}
chunker_mk   = {c: mk for c, mk in zip(df_pareto['Chunker'].unique(), ['o', 's', '^'])}

fig, ax = plt.subplots(figsize=(11, 7))
for llm in llm_order_p:
    for chunker, marker in chunker_mk.items():
        sub   = df_pareto[(df_pareto['LLM'] == llm) & (df_pareto['Chunker'] == chunker)]
        non_p = sub[~sub['pareto']]
        p     = sub[sub['pareto']]
        if not non_p.empty:
            ax.scatter(non_p['latency'], non_p['avg_quality'],
                       color=llm_colors_p[llm], marker=marker, s=65,
                       alpha=0.40, edgecolors='k', linewidths=0.3)
        if not p.empty:
            ax.scatter(p['latency'], p['avg_quality'],
                       color=llm_colors_p[llm], marker=marker, s=150,
                       alpha=0.95, edgecolors='k', linewidths=1.4, zorder=5)

pf = df_pareto[df_pareto['pareto']].sort_values('latency')
ax.step(pf['latency'], pf['avg_quality'], where='post',
        color='black', linestyle='--', linewidth=1.5, label='Pareto front', zorder=4)
for _, row in pf.iterrows():
    ax.annotate(
        row['Configuration'].replace(' | ', '\n'),
        xy=(row['latency'], row['avg_quality']),
        xytext=(6, 4), textcoords='offset points',
        fontsize=6.5, bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75),
    )

ax.axhline(0.75, color='green',      linestyle=':', alpha=0.75, lw=1.5, label='Quality floor (0.75)')
ax.axvline(15.0, color='darkorange', linestyle=':', alpha=0.75, lw=1.5, label='Latency ceiling (15s)')

llm_patches = [mpatches.Patch(color=llm_colors_p[l], label=l) for l in llm_order_p]
mk_patches  = [plt.scatter([], [], marker=mk, color='grey', s=70, label=ch)
               for ch, mk in chunker_mk.items()]
extra_lines = [
    plt.Line2D([0], [0], color='black',     linestyle='--', label='Pareto front'),
    plt.Line2D([0], [0], color='green',      linestyle=':', label='Quality >= 0.75'),
    plt.Line2D([0], [0], color='darkorange', linestyle=':', label='Latency <= 15s'),
]
ax.legend(handles=llm_patches + mk_patches + extra_lines, fontsize=7.5, loc='lower right')
ax.set_xlabel('Mean Latency (s)')
ax.set_ylabel('Avg. RAGAS quality score')
ax.set_title('Pareto frontier – Quality vs Latency\nLarge opaque markers = Pareto-optimal configs')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('pareto_frontier.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPareto-optimal configurations (non-dominated on quality up and latency down):')
print(df_pareto[df_pareto['pareto']][['Configuration', 'Chunker', 'Embedder', 'LLM',
                                      'avg_quality', 'latency']].to_string(index=False))

## 12  Statistical Significance – Kruskal-Wallis & Mann-Whitney U

### Why Kruskal-Wallis instead of one-way ANOVA?

ANOVA assumes that residuals are **normally distributed** and that group variances are equal
(homoscedasticity). RAGAS metrics are bounded in [0, 1], frequently pile up at 1.0
(see violin plots in Section 7 for faithfulness and source attribution), and per-question
distributions are visibly non-normal. These violations make ANOVA unreliable here.

The **Kruskal-Wallis H-test** is the non-parametric analogue of one-way ANOVA:
- Makes **no normality or variance-homogeneity assumption** – operates on ranks.
- Robust to outliers and ceiling effects (common in bounded metrics).
- Tests: 'does at least one group come from a different distribution?'
- **Limitation:** a significant result only says *some* groups differ; it does not identify which.

### Why Mann-Whitney U for pairwise comparisons?

When Kruskal-Wallis is significant, **pairwise Mann-Whitney U tests** with **Bonferroni
correction** identify which specific pairs of component values differ significantly.
Mann-Whitney U is the non-parametric equivalent of an independent-samples t-test –
it also ranks observations and therefore shares the same robustness properties.

**Why Bonferroni correction?** With 3 component values there are 3 pairwise tests per metric.
Running multiple tests inflates the family-wise error rate; Bonferroni is the simplest
correction (multiply each p-value by the number of tests, cap at 1.0).

**Effect size** is reported as rank-biserial correlation *r*:
- |r| < 0.10 = negligible, 0.10–0.30 = small, 0.30–0.50 = medium, > 0.50 = large.

**Reading the heatmap:** darker blue = smaller p-value = more significant difference.
A tick (checkmark) in the table means the component choice statistically matters for that metric.

In [ ]:
# 12a  Kruskal-Wallis omnibus tests
sig_metrics = [m for m in RAGAS_METRICS + ['source_attribution', 'latency'] if m in df.columns]

rows_kw = []
for comp in COMPONENTS:
    for metric in sig_metrics:
        groups = [g[metric].dropna().values for _, g in df.groupby(comp)]
        if all(len(g) > 1 for g in groups):
            stat, pval = stats.kruskal(*groups)
            rows_kw.append({
                'Component': comp,
                'Metric': METRIC_LABELS.get(metric, metric),
                'H-statistic': round(stat, 3),
                'p-value': round(pval, 6),
                'alpha=0.05': 'significant' if pval < 0.05 else 'n.s.',
                'Effect': 'large' if pval < 0.001 else ('medium' if pval < 0.01 else ('small' if pval < 0.05 else 'none')),
            })

df_kw = pd.DataFrame(rows_kw)
print('Kruskal-Wallis omnibus tests')
print('=' * 70)
display(df_kw)

pval_pivot = df_kw.pivot(index='Component', columns='Metric', values='p-value')
log_pvals  = -np.log10(pval_pivot.clip(lower=1e-10))
fig, ax = plt.subplots(figsize=(max(8, 2.2 * len(sig_metrics)), 3.5))
sns.heatmap(
    log_pvals,
    annot=pval_pivot.map(lambda v: f'{v:.4f}' if not pd.isna(v) else '-'),
    fmt='', cmap='Blues', linewidths=0.5, ax=ax,
    cbar_kws={'label': '-log10(p-value)  [darker = more significant]'},
)
ax.set_title(
    'Statistical significance of component choices (Kruskal-Wallis)\n'
    'Cell values are raw p-values; colour intensity = -log10(p)',
    pad=10,
)
plt.tight_layout()
plt.savefig('significance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 12b  Pairwise Mann-Whitney U (post-hoc, Bonferroni corrected)
rows_mw = []
for comp in COMPONENTS:
    levels   = df[comp].unique()
    n_pairs  = len(list(combinations(levels, 2)))
    for metric in sig_metrics:
        for a, b in combinations(levels, 2):
            g_a = df.loc[df[comp] == a, metric].dropna().values
            g_b = df.loc[df[comp] == b, metric].dropna().values
            if len(g_a) > 1 and len(g_b) > 1:
                u, pval = stats.mannwhitneyu(g_a, g_b, alternative='two-sided')
                pval_c  = min(pval * n_pairs, 1.0)
                r_rb    = 1 - 2 * u / (len(g_a) * len(g_b))
                rows_mw.append({
                    'Component': comp,
                    'Metric':    METRIC_LABELS.get(metric, metric),
                    'Group A':   str(a).split('/')[-1][:25],
                    'Group B':   str(b).split('/')[-1][:25],
                    'U':         round(u, 1),
                    'p (raw)':   round(pval,   5),
                    'p (Bonf.)': round(pval_c, 5),
                    'Sig.':      'Yes' if pval_c < 0.05 else 'No',
                    'r_rb':      round(r_rb, 3),
                    'Effect':    'large' if abs(r_rb) > 0.5 else ('medium' if abs(r_rb) > 0.3 else ('small' if abs(r_rb) > 0.1 else 'negligible')),
                })

df_mw = pd.DataFrame(rows_mw)
print('Pairwise Mann-Whitney U (Bonferroni-corrected) – significant pairs only:')
display(df_mw[df_mw['Sig.'] == 'Yes'].sort_values(['Component', 'Metric']))

# 12c  Box plots with threshold lines per component
for metric in sig_metrics:
    fig, axes = plt.subplots(1, len(COMPONENTS), figsize=(5 * len(COMPONENTS), 5))
    for col_j, comp in enumerate(COMPONENTS):
        ax    = axes[col_j]
        order = df.groupby(comp)[metric].median().sort_values(ascending=False).index.tolist()
        sns.boxplot(data=df, x=comp, y=metric, order=order, palette='Set2', ax=ax)
        if metric in THRESHOLDS:
            thr = THRESHOLDS[metric]
            ax.axhline(thr['good'],       color='green',      linestyle='--', alpha=0.7, lw=1.2,
                       label=f"Good ({thr['good']})")
            ax.axhline(thr['acceptable'], color='darkorange', linestyle='--', alpha=0.7, lw=1.2,
                       label=f"Acceptable ({thr['acceptable']})")
        kw_row = df_kw[(df_kw['Component'] == comp) & (df_kw['Metric'] == METRIC_LABELS.get(metric, metric))]
        kw_label = f"KW p={kw_row.iloc[0]['p-value']:.4f} (sig.)" if not kw_row.empty and kw_row.iloc[0]['p-value'] < 0.05 else 'KW n.s.'
        ax.set_xlabel(kw_label, fontsize=8)
        ax.set_title(comp, fontsize=10)
        ax.set_ylabel(METRIC_LABELS.get(metric, metric) if col_j == 0 else '')
        ax.tick_params(axis='x', labelrotation=15, labelsize=8)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4)
        if col_j == 0:
            ax.legend(fontsize=7.5)
    fig.suptitle(
        f'{METRIC_LABELS.get(metric, metric)} by component'
        '\n(dashed = good/acceptable thresholds; x-axis label = Kruskal-Wallis result)',
        fontsize=12,
    )
    plt.tight_layout()
    plt.savefig(f'boxplot_{metric}_components.png', dpi=150, bbox_inches='tight')
    plt.show()

## 13  Decision Framework & Weighted Ranking

### Why a weighted composite score instead of a simple RAGAS mean?

All four RAGAS metrics are not equally important for an ESG reporting pipeline:

| Metric | Weight | Rationale |
|---|---|---|
| Faithfulness | 25% | Most critical – a hallucinated ESG claim is legally and reputationally damaging |
| Source Attribution | 20% | ESG traceability requirement: every claim must be traceable to a source |
| Answer Relevancy | 20% | The answer must directly address the compliance question |
| Context Recall | 20% | The retriever must surface the relevant evidence from the report |
| Context Precision | 15% | Noisy chunks add cost but the LLM can partly filter irrelevance |

A **latency benefit score** (1 = fastest, 0 = slowest, normalised within the observed range)
is blended at **20% weight** alongside quality (80%), reflecting that speed matters
for an interactive tool but should not override correctness in a compliance context.

### Threshold-based compliance bars

The final chart shows what fraction of questions each component value pushes above the
*good* threshold across all metrics simultaneously – a single-number reliability score
that complements the latency-weighted ranking.

In [ ]:
# 13a  Weighted composite score & ranking
WEIGHTS = {
    'faithfulness':       0.25,
    'source_attribution': 0.20,
    'answer_relevancy':   0.20,
    'context_recall':     0.20,
    'context_precision':  0.15,
}

df_dec = df_configs.copy()
lat_min, lat_max = df_dec['latency'].min(), df_dec['latency'].max()
df_dec['latency_score'] = 1 - (df_dec['latency'] - lat_min) / (lat_max - lat_min + 1e-9)

qual_w  = {m: WEIGHTS[m] for m in WEIGHTS if m in df_dec.columns}
total_w = sum(qual_w.values())
df_dec['weighted_quality'] = sum(df_dec[m] * (w / total_w) for m, w in qual_w.items())
df_dec['combined_score']   = 0.80 * df_dec['weighted_quality'] + 0.20 * df_dec['latency_score']
df_dec = df_dec.sort_values('combined_score', ascending=False).reset_index(drop=True)
df_dec['rank'] = df_dec.index + 1

show_cols = [c for c in ['rank', 'Configuration', 'weighted_quality', 'latency', 'latency_score', 'combined_score'] if c in df_dec.columns]
print('Decision Framework Leaderboard  (80% weighted quality + 20% speed)')
display(df_dec[show_cols].round(4))

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, max(6, 0.38 * len(df_dec))))
bar_colors = [PALETTE[i % 10] for i in range(len(df_dec) - 1, -1, -1)]
hbars = ax.barh(df_dec['Configuration'][::-1], df_dec['combined_score'][::-1],
                color=bar_colors, alpha=0.82, edgecolor='white')
mean_sc = df_dec['combined_score'].mean()
ax.axvline(mean_sc, color='black', linestyle='--', lw=1.3, label=f'Mean ({mean_sc:.3f})')
for bar, val in zip(hbars, df_dec['combined_score'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=7.5)
ax.set_xlabel('Combined score  (80% weighted quality + 20% speed)')
ax.set_title('Overall Configuration Ranking', pad=10)
ax.legend(fontsize=9)
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('ranking_combined_score.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 13b  Component recommendation text + threshold compliance grouped bar chart
print('=' * 70)
print('  COMPONENT RECOMMENDATION SUMMARY')
print('=' * 70)
for comp in COMPONENTS:
    grp = dfs_comp[comp].copy()
    ragas_g = [m for m in RAGAS_METRICS if m in grp.columns]
    grp['q_score'] = grp[ragas_g].mean(axis=1) if ragas_g else grp['source_attribution']
    best_q = grp.loc[grp['q_score'].idxmax(), comp]
    best_l = grp.loc[grp['latency'].idxmin(), comp]
    print(f'\n  {comp}')
    print(f'  {"-" * 55}')
    for _, row in grp.sort_values('q_score', ascending=False).iterrows():
        flags = []
        if row[comp] == best_q: flags.append('best quality')
        if row[comp] == best_l: flags.append('best speed')
        print(f"    {str(row[comp]).split('/')[-1]:<35}  "
              f"quality={row['q_score']:.3f}  latency={row['latency']:.1f}s"
              + (f"  <-- {', '.join(flags)}" if flags else ''))

# Threshold compliance grouped bar chart
thresh_show = [m for m in THRESHOLDS if m in df.columns]
fig, axes   = plt.subplots(1, len(COMPONENTS), figsize=(6 * len(COMPONENTS), 5))
for col_j, comp in enumerate(COMPONENTS):
    ax        = axes[col_j]
    comp_vals = df[comp].unique()
    x         = np.arange(len(thresh_show))
    width     = 0.8 / len(comp_vals)
    for i, val in enumerate(comp_vals):
        sub  = df[df[comp] == val]
        pcts = [(sub[m].apply(lambda v: classify(v, m)) == 'good').mean() for m in thresh_show]
        ax.bar(x + i * width - (len(comp_vals) - 1) * width / 2, pcts, width,
               label=str(val).split('/')[-1][:20], alpha=0.82, edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels([METRIC_LABELS.get(m, m) for m in thresh_show],
                       rotation=20, ha='right', fontsize=8)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Fraction meeting 'good' threshold" if col_j == 0 else '')
    ax.set_title(comp, fontsize=10)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)
    ax.legend(fontsize=7, loc='lower right')
fig.suptitle("Threshold compliance by component  ('good' tier pass rate per metric)", fontsize=12)
plt.tight_layout()
plt.savefig('barplot_threshold_compliance.png', dpi=150, bbox_inches='tight')
plt.show()

## 14  Export combined per-question CSV

One row per (question x configuration) with every computed metric.

In [ ]:
export_cols = (
    [c for c in ['question_id', 'question', 'Chunker', 'Embedder', 'LLM', 'Configuration'] if c in df_raw.columns]
    + [m for m in ALL_METRICS if m in df_raw.columns]
)
df_export = df_raw[export_cols].copy().round(6)
export_path = 'per_question_all_metrics.csv'
df_export.to_csv(export_path, index=False, sep=';')
print(f'Saved {len(df_export)} rows to {export_path}')
df_export